# Conditional Proximity-Gated Nanobody — Design Walkthrough

## Concept

A nanobody is tethered to an antibody via a flexible (G₄S)ₙ linker.  
The nanobody can **only** bind its target when the antibody is anchored to a specific antigen — creating an **AND-gate** conditional binding behaviour.

```
         [Nanobody]
              |
       ~flexible linker~
              |
  [VH2─VL2 / VH1─VL1]       ← antibody fragment
              |
          [Antigen] ··membrane·· [Target]
```

### Key biophysical principle

When the antibody anchors to the antigen, the flexible linker confines the nanobody to a **small search volume** directly above the membrane surface.  
The **effective local concentration** of the nanobody near the target is given by the polymer probability density at that distance:

$$c_\text{eff}(d) = \left(\frac{3}{2\pi \langle r^2 \rangle}\right)^{3/2} \exp\!\left(-\frac{3d^2}{2\langle r^2 \rangle}\right) \cdot \frac{1}{N_A}$$

This can reach **µM range** — far above the nM Kd of typical nanobodies — driving near-complete conditional binding.

In [ ]:
import sys
sys.path.insert(0, '..')  # allow running from notebooks/ directory

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from binary_antibodies import LinkerModel, ConditionalConstruct, LinkerSequence
from binary_antibodies.sequences import recommended_linkers

%matplotlib inline
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120,
                     'axes.spines.top': False, 'axes.spines.right': False})

## 1. Linker physics — how large is the search volume?

We model the linker as a **freely-jointed chain (FJC)** — each Cα–Cα virtual bond
is treated as an independent segment of length `l = 0.38 nm`.  
The end-to-end distribution is Gaussian:

$$P(r) = 4\pi r^2 \left(\frac{3}{2\pi N l^2}\right)^{3/2} \exp\!\left(-\frac{3r^2}{2Nl^2}\right)$$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#ff7f0e']
lengths = [20, 40, 60, 80, 120]
r = np.linspace(0.01, 55, 500)

ax = axes[0]
for color, n in zip(colors, lengths):
    lm = LinkerModel(n_residues=n)
    ax.plot(r, lm.end_to_end_pdf(r), color=color, lw=2,
            label=f"{n} res  (Lc={lm.contour_length_nm:.0f} nm, "
                  f"r_rms={lm.rms_end_to_end_nm:.1f} nm)")

ax.set_xlabel('End-to-end distance r (nm)')
ax.set_ylabel('Probability density P(r)  [nm⁻¹]')
ax.set_title('FJC end-to-end distance distribution')
ax.legend(fontsize=8)

ax = axes[1]
distances = np.linspace(0.1, 25, 300)
for color, n in zip(colors, lengths):
    lm = LinkerModel(n_residues=n)
    c_eff = np.array([lm.effective_concentration_M(d) * 1e6 for d in distances])
    ax.semilogy(distances, np.where(c_eff > 0, c_eff, 1e-12),
                color=color, lw=2, label=f"{n} res")

ax.axhline(0.01, color='black', linestyle='--', lw=1.5, label='10 nM Kd')
ax.set_xlabel('Antigen–target distance (nm)')
ax.set_ylabel('Effective [nanobody] (µM)')
ax.set_title('Effective local nanobody concentration')
ax.legend(fontsize=8)
ax.set_ylim(bottom=1e-10)

fig.tight_layout()
plt.show()

## 2. Define a concrete construct

| Component | Kd |
|---|---|
| Antibody (scFv) for antigen | 1 nM |
| Nanobody for target | 10 nM |

Antigen–target geometry: **8 nm** (a typical distance between two adjacent
membrane proteins, e.g. within the same receptor complex).

In [ ]:
DISTANCE_NM = 8.0          # antigen–target distance on the membrane
KD_ANTIBODY = 1e-9         # 1 nM
KD_NANOBODY = 10e-9        # 10 nM

construct = ConditionalConstruct(
    antibody_kd_M=KD_ANTIBODY,
    nanobody_kd_M=KD_NANOBODY,
    linker=LinkerModel(n_residues=60),
    name="Anti-antigen scFv – (G4S)12 – anti-target nanobody",
)

print(construct.summary(DISTANCE_NM))

## 3. Linker optimisation

Find the **minimum linker length** that achieves ≥ 50% nanobody occupancy
when the antibody is anchored at 8 nm from the target.

In [ ]:
opt_n = construct.optimal_linker_length(
    distance_nm=DISTANCE_NM,
    target_occupancy=0.50,
)
print(f"Minimum linker length for ≥50% occupancy at {DISTANCE_NM} nm: {opt_n} residues")

# Recommend the equivalent (G4S)n construct
recs = recommended_linkers(
    distance_nm=DISTANCE_NM,
    nanobody_kd_M=KD_NANOBODY,
    target_occupancy=0.50,
)
best = recs[0]
print(f"\nRecommended (G4S)n linker:")
print(f"  ({best['n_repeats']} repeats = {best['n_residues']} residues)")
print(f"  Sequence   : {best['sequence']}")
print(f"  Occupancy  : {best['occupancy']:.1%}")
print(f"  Molar mass : {best['molecular_weight_Da']:.0f} Da")

## 4. Occupancy vs linker length — the design curve

In [ ]:
scan = construct.scan_linker_lengths(distance_nm=DISTANCE_NM,
                                     lengths=list(range(5, 151, 5)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(scan['n_residues'], scan['occupancy'], 'o-', color='#1f77b4', ms=4, lw=2)
ax.axhline(0.5, color='black', linestyle='--', lw=1, label='50% target')
if opt_n:
    lm_opt = LinkerModel(n_residues=opt_n)
    occ_opt = lm_opt.occupancy(KD_NANOBODY, DISTANCE_NM)
    ax.axvline(opt_n, color='red', linestyle=':', lw=1.5, label=f'Optimal = {opt_n} res')
    ax.scatter([opt_n], [occ_opt], color='red', zorder=5, s=80)
ax.set_xlabel('Linker length (residues)')
ax.set_ylabel('Nanobody occupancy (anchored)')
ax.set_title(f'Conditional occupancy vs linker length\n(antigen–target distance = {DISTANCE_NM} nm)')
ax.set_ylim(0, 1.05)
ax.legend()

ax = axes[1]
ax.semilogy(scan['n_residues'], np.where(scan['c_eff_M'] > 0, scan['c_eff_M'] * 1e6, 1e-12),
            'o-', color='#2ca02c', ms=4, lw=2)
ax.axhline(KD_NANOBODY * 1e6, color='black', linestyle='--', lw=1,
           label=f'Nanobody Kd = {KD_NANOBODY*1e9:.0f} nM')
ax.set_xlabel('Linker length (residues)')
ax.set_ylabel('Effective [nanobody] (µM)')
ax.set_title('Effective local concentration vs linker length')
ax.legend()

fig.tight_layout()
plt.show()

## 5. 2-D Parameter Space: distance × linker length

In [ ]:
distances_nm = np.linspace(1, 22, 50)
linker_lengths = np.arange(10, 151, 5, dtype=int)

data = construct.parameter_space_heatmap(
    distances_nm=distances_nm,
    linker_lengths=linker_lengths,
)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(
    data['occupancy'],
    origin='lower', aspect='auto', cmap='RdYlGn', vmin=0, vmax=1,
    extent=[linker_lengths[0], linker_lengths[-1], distances_nm[0], distances_nm[-1]],
)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label('Nanobody occupancy (anchored)', fontsize=10)

# 50% and 90% contours
cs = ax.contour(linker_lengths, distances_nm, data['occupancy'],
                levels=[0.5, 0.9], colors=['white', 'yellow'], linewidths=2,
                linestyles=['--', ':'])
ax.clabel(cs, fmt={0.5: '50%', 0.9: '90%'}, fontsize=9)

ax.scatter([opt_n], [DISTANCE_NM], marker='*', color='cyan', s=200, zorder=5,
           label=f'Optimal point ({opt_n} res, {DISTANCE_NM} nm)')

ax.set_xlabel('Linker length (residues)')
ax.set_ylabel('Antigen–target distance (nm)')
ax.set_title(f'Conditional nanobody occupancy heatmap\n(Nanobody Kd = {KD_NANOBODY*1e9:.0f} nM)')
ax.legend(fontsize=9, loc='upper left')

fig.tight_layout()
plt.show()

## 6. AND-gate selectivity: anchored vs free

The **selectivity ratio** quantifies how much more the nanobody binds the target
when the antibody is anchored vs when it is freely diffusing in solution.

In [ ]:
opt_construct = ConditionalConstruct(
    antibody_kd_M=KD_ANTIBODY,
    nanobody_kd_M=KD_NANOBODY,
    linker=LinkerModel(n_residues=opt_n),
)

# Vary the bulk concentration of soluble nanobody construct (free state)
bulk_concs = np.logspace(-12, -6, 100)  # 1 pM – 1 µM
ratios = [opt_construct.selectivity_ratio(DISTANCE_NM, c) for c in bulk_concs]

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(bulk_concs * 1e9, ratios, color='#9467bd', lw=2)
ax.axvline(KD_NANOBODY * 1e9, color='black', linestyle='--', lw=1,
           label=f'Nanobody Kd = {KD_NANOBODY*1e9:.0f} nM')
ax.set_xlabel('Bulk construct concentration (nM)')
ax.set_ylabel('Selectivity ratio (anchored / free)')
ax.set_title('AND-gate selectivity: how conditional is the binding?')
ax.legend()
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())

fig.tight_layout()
plt.show()

# Print key values
print(f"Anchored nanobody occupancy:  {opt_construct.nanobody_occupancy_anchored(DISTANCE_NM):.1%}")
print(f"Free nanobody occupancy (1 nM bulk): {opt_construct.nanobody_occupancy_free(1e-9):.4%}")
print(f"Selectivity ratio at 1 nM bulk: {opt_construct.selectivity_ratio(DISTANCE_NM, 1e-9):.1f}×")

## 7. Design recommendations

### Rules of thumb

| Antigen–target distance | Minimum (G₄S)ₙ repeats for ≥50% occupancy |
|---|---|
| 5 nm  | ~6 repeats (30 res) |
| 8 nm  | ~9 repeats (45 res) |
| 12 nm | ~14 repeats (70 res) |
| 15 nm | ~18 repeats (90 res) |

### Practical design workflow

1. **Identify the antigen and target** on the same cell surface.
2. **Estimate or measure the antigen–target distance** (cryo-EM, MD simulation, or structural homology).
3. **Select a nanobody** with intrinsic Kd ≪ C_eff (typically 1–100 nM works well).
4. **Compute the minimum linker length** using `ConditionalConstruct.optimal_linker_length()`.
5. **Round up to the nearest (G₄S)ₙ repeat** for practical cloning.
6. **Validate experimentally**: SPR or bio-layer interferometry on surface-tethered antigen, with and without the antibody arm engaged.

### Construct architecture (N→C)

```
SP – [VH1] – [VL1] – (G4S)3 – [VH2] – [VL2] – (G4S)n – [Nanobody VHH]
```

- `SP`: signal peptide for secretion
- `[VH1]–[VL1]`: antibody heavy/light chain variable domains (scFv format, antigen arm)
- `(G4S)3`: short interdomain linker between the two scFv halves
- `[VH2]–[VL2]`: second scFv for bivalent antigen binding (optional)
- `(G4S)n`: the conditional flexible linker (length from optimisation)
- `[Nanobody VHH]`: single-domain antibody targeting the membrane protein of interest